# Day I14 — 측정 신뢰도와 타당도: Bland–Altman과 ICC를 손으로 + 2주차 정리

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I14.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I14.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 상관, Bland–Altman, ICC, SEM을 한 번에

In [ ]:
import glob
import numpy as np, pandas as pd
from scipy import stats
from gait_events import strides

fs = 100
rng = np.random.default_rng(14)
files = sorted(glob.glob("data/gaitpdb/*_01.txt"))
pick = rng.choice(files, 30, replace=False)       # 참가자 30명(무작위, seed 고정)

ref = []
for p in pick:
    r, ok = strides(np.loadtxt(p)[:, 17], 20, fs)  # 왼발, 20 N 문턱 = 기준 방법
    ref.append(r[ok, 2].mean())                    # 참가자별 평균 stance time(s)
ref = np.array(ref)
# 가상 IMU 방법: 체계적 차이(bias) +20 ms, 참가자마다 다른 무작위 오차 SD 15 ms
imu = ref + 0.020 + rng.normal(0, 0.015, len(ref))

# ---- 상관 ----
r = np.corrcoef(ref, imu)[0, 1]
print(f"참가자 {len(ref)}명, 기준 stance 범위 {ref.min():.2f}–{ref.max():.2f} s")
print(f"Pearson r = {r:.3f}")

# ---- Bland–Altman ----
diff = imu - ref
mean2 = (imu + ref) / 2
n = len(diff)
bias, s = diff.mean(), diff.std(ddof=1)
print(f"bias(IMU − 기준) = {1000 * bias:.1f} ms, 차이의 SD = {1000 * s:.1f} ms")
print(f"일치 한계 bias ± 1.96·SD = [{1000 * (bias - 1.96 * s):.1f}, {1000 * (bias + 1.96 * s):.1f}] ms")
print(f"일치 한계 bias ± 2·SD    = [{1000 * (bias - 2 * s):.1f}, {1000 * (bias + 2 * s):.1f}] ms (1986년 원 논문 방식)")
tcrit = stats.t.ppf(0.975, n - 1)
se_bias, se_loa = s / np.sqrt(n), np.sqrt(3 * s ** 2 / n)
print(f"bias의 95% CI = [{1000 * (bias - tcrit * se_bias):.1f}, {1000 * (bias + tcrit * se_bias):.1f}] ms")
print(f"아래 한계의 95% CI = [{1000 * (bias - 1.96 * s - tcrit * se_loa):.1f}, {1000 * (bias - 1.96 * s + tcrit * se_loa):.1f}] ms")
print(f"차이와 평균의 상관 r = {np.corrcoef(mean2, diff)[0, 1]:.2f} (차이가 크기에 따라 변하는지 점검)")

# ---- ICC: 이원 분산분석의 평균제곱으로 직접 ----
def icc_table(Y):
    """Y: (참가자 n, 방법/평가자 k). 평균제곱 MSR(행), MSC(열), MSE(잔차)를 돌려준다."""
    n, k = Y.shape
    grand = Y.mean()
    ss_r = k * ((Y.mean(axis=1) - grand) ** 2).sum()
    ss_c = n * ((Y.mean(axis=0) - grand) ** 2).sum()
    ss_e = ((Y - grand) ** 2).sum() - ss_r - ss_c
    return ss_r / (n - 1), ss_c / (k - 1), ss_e / ((n - 1) * (k - 1)), n, k

def icc_2_1(Y):            # 2원, 절대 일치(absolute agreement), 단일 측정
    msr, msc, mse, n, k = icc_table(Y)
    return (msr - mse) / (msr + (k - 1) * mse + k / n * (msc - mse))

def icc_3_1(Y):            # 2원, 일관성(consistency), 단일 측정
    msr, msc, mse, n, k = icc_table(Y)
    return (msr - mse) / (msr + (k - 1) * mse)

Y = np.column_stack([ref, imu])
msr, msc, mse, _, _ = icc_table(Y)
print(f"\nMSR = {msr:.6f}, MSC = {msc:.6f}, MSE = {mse:.6f}")
print(f"ICC(2,1) 절대 일치 = {icc_2_1(Y):.3f}")
print(f"ICC(3,1) 일관성   = {icc_3_1(Y):.3f}")
Y0 = np.column_stack([ref, imu - bias])                 # 같은 무작위 오차, bias만 제거
print(f"bias가 없을 때 ICC(2,1) = {icc_2_1(Y0):.3f}")

# 자기 점검(assert): 두 방법이 같으면 1, 상수만 더하면 일관성은 그대로 1
assert np.isclose(icc_2_1(np.column_stack([ref, ref])), 1)
assert np.isclose(icc_3_1(np.column_stack([ref, ref + 0.05])), 1)

# ---- bootstrap 95% CI (참가자 단위) ----
bs = [icc_2_1(Y[rng.integers(0, n, n)]) for _ in range(5000)]
print(f"ICC(2,1) bootstrap 95% CI = [{np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f}]")

# ---- SEM, MDC ----
icc = icc_2_1(Y)
sd_all = Y.std(ddof=1)
sem = sd_all * np.sqrt(1 - icc)
print(f"SEM = SD·√(1 − ICC) = {1000 * sem:.1f} ms, MDC95 = 1.96·√2·SEM = {1000 * 1.96 * np.sqrt(2) * sem:.1f} ms")